# GroupIndex Benchmark

Benchmark different GroupIndex implementations across varying dataset sizes.

Results are stored per-implementation in `results/gi_<name>.json` so you can run
benchmarks incrementally - add a new GroupIndex type and benchmark only that one.
Re-running an existing type overwrites its previous results.

The summary cells at the end aggregate all result files found in `results/`.

In [ ]:
import json
import pathlib
import time

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from skrough.algorithms.reducts import get_approx_reduct_greedy_heuristic
from skrough.disorder_measures import gini_impurity

In [ ]:
# Configuration
# Edit GROUP_INDEX_TO_RUN to benchmark only specific implementations.
# Leave as-is to run all.

DATA_DIR = pathlib.Path("../../data/local/seismic")
RESULTS_DIR = pathlib.Path("results")
FIGURES_DIR = pathlib.Path("figures")

DATASET_SIZES = [1000, 2000, 5000, 10000, 20000, 25000, 50000, 75000, 100000, 133150]
GROUP_INDEX_TO_RUN = ["lazy", "dict", "hash", "pure", "numba"]

CANDIDATES_COUNT = 20
N_REDUCTS = 10
N_JOBS = -1
EPSILON = 0.2

TIMEOUT_SECONDS = 120  # 2 minutes

In [ ]:
# Load and encode the full dataset

data = pd.read_csv(DATA_DIR / "seismic_training_data_features_discretized.csv")
target = pd.read_csv(DATA_DIR / "seismic_training_labels.csv", header=None)[0]

data.drop("main_working_id", axis=1, inplace=True)

data_encoded = OrdinalEncoder().fit_transform(data)
target_encoded = LabelEncoder().fit_transform(target)

print(f"Full dataset shape: {data_encoded.shape}")
print(f"Total objects: {len(data_encoded)}")

In [ ]:
# Benchmark runner
# Runs only the implementations listed in GROUP_INDEX_TO_RUN.
# Each implementation's results are saved to results/gi_<name>.json,
# overwriting any previous results for that implementation.

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for gi_name in GROUP_INDEX_TO_RUN:
    print(f"\n{'=' * 60}")
    print(f"GroupIndex: {gi_name}")
    print(f"{'=' * 60}")

    gi_results = {}
    timed_out = False

    for size in DATASET_SIZES:
        if timed_out:
            print(f"  [size={size}] SKIPPED (timed out at smaller size)")
            gi_results[size] = {
                "status": "skipped",
                "reason": "timeout_at_smaller_size",
            }
            continue

        print(f"  [size={size}] Running...", end=" ", flush=True)

        actual_size = min(size, len(data_encoded))
        x_subset, _, y_subset, _ = train_test_split(
            data_encoded, target_encoded, train_size=actual_size, random_state=42
        )

        start = time.perf_counter()

        try:
            get_approx_reduct_greedy_heuristic(
                x=x_subset,
                y=y_subset,
                disorder_fun=gini_impurity,
                epsilon=EPSILON,
                candidates_count=CANDIDATES_COUNT,
                n_reducts=N_REDUCTS,
                n_jobs=N_JOBS,
                group_index_class=gi_name,
            )
            elapsed = time.perf_counter() - start
            print(f"{elapsed:.2f}s")

            gi_results[size] = {
                "status": "completed",
                "time_seconds": round(elapsed, 3),
                "actual_size": actual_size,
            }

            if elapsed > TIMEOUT_SECONDS:
                print(f"  [{gi_name}] TIMEOUT ({elapsed:.2f}s > {TIMEOUT_SECONDS}s)")
                timed_out = True

        except Exception as e:
            elapsed = time.perf_counter() - start
            print(f"ERROR ({elapsed:.2f}s): {e}")
            gi_results[size] = {
                "status": "error",
                "time_seconds": round(elapsed, 3),
                "error": str(e),
            }

    # Save per-implementation results
    output_file = RESULTS_DIR / f"gi_{gi_name}.json"
    with open(output_file, "w") as f:
        json.dump(gi_results, f, indent=2)
    print(f"  Results saved to {output_file}")

In [ ]:
# Load all results from results/ directory
# Aggregates all gi_*.json files into a single results dict.

results = {}
all_gi_names = []

for result_file in sorted(RESULTS_DIR.glob("gi_*.json")):
    gi_name = result_file.stem.removeprefix("gi_")
    all_gi_names.append(gi_name)

    with open(result_file) as f:
        gi_results = json.load(f)

    for size_str, entry in gi_results.items():
        size = int(size_str)
        if size not in results:
            results[size] = {}
        results[size][gi_name] = entry

all_gi_names = sorted(set(all_gi_names))
print(f"Loaded results for: {all_gi_names}")
print(f"Dataset sizes: {sorted(results.keys())}")

In [ ]:
# Summary table

summary_rows = []
for size in sorted(results.keys()):
    for gi_name in all_gi_names:
        entry = results.get(size, {}).get(gi_name, {})
        summary_rows.append(
            {
                "dataset_size": size,
                "group_index": gi_name,
                "status": entry.get("status", "not_run"),
                "time_seconds": entry.get("time_seconds", None),
            }
        )

summary_df = pd.DataFrame(summary_rows)
pivot = summary_df.pivot(
    index="dataset_size", columns="group_index", values="time_seconds"
)
print("\nTime (seconds) by dataset size and GroupIndex implementation:")
print(pivot.to_string())

In [ ]:
# Plot: time vs. dataset size for each GroupIndex implementation

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))

for gi_name in all_gi_names:
    sizes = []
    times = []
    for size in sorted(results.keys()):
        entry = results.get(size, {}).get(gi_name, {})
        if entry.get("status") == "completed":
            sizes.append(entry.get("actual_size", size))
            times.append(entry["time_seconds"])
    if sizes:
        ax.plot(sizes, times, marker="o", label=gi_name)

ax.set_xlabel("Number of objects")
ax.set_ylabel("Time (seconds)")
ax.set_title("GroupIndex Benchmark: Time vs. Dataset Size")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "group_index_benchmark_plot.png", dpi=150)
plt.show()